# 03 WeightWatcher Analysis

In [ ]:
# User-editable papermill/environment configuration
from pathlib import Path
import os

DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/tmp/wwpgd_v2/data"))
RESULTS_ROOT = Path(
    os.environ.get(
        "WWGPT_RESULTS_ROOT",
        os.environ.get(
            "RESULTS_ROOT",
            "/tmp/wwpgd_v2/real_level0_results_v5/"
            "experiments/level_00/multiplier_20",
        ),
    )
)
RUN_LOG = Path(os.environ.get("RUN_LOG", "/tmp/wwpgd_v2/real_level0_five_seed_v4.log"))
PID_FILE = Path(os.environ.get("PID_FILE", "/tmp/wwpgd_v2/real_level0_five_seed_v4.pid"))

ANALYSIS_DIR = RESULTS_ROOT / "analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
INCLUDE_LEGACY = False
EXPECTED_SEEDS = {1337, 2027, 4099, 7919, 104729}


In [ ]:
import sys
from pathlib import Path
cwd = Path.cwd().resolve()
repo = cwd if (cwd/'src'/'wwgpt').exists() else cwd.parent
if str(repo/'src') not in sys.path:
    sys.path.insert(0, str(repo/'src'))
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from IPython.display import display
from scipy import stats
from wwgpt.analysis import *
RESULTS_ROOT = resolve_experiment_root(RESULTS_ROOT)
ANALYSIS_DIR = RESULTS_ROOT / 'analysis'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
print(f'Repository root: {repo}')
print(f'Results root: {RESULTS_ROOT}')
print(f'Data root: {DATA_ROOT}')
print(f'Run log: {RUN_LOG}')
print(f'PID file: {PID_FILE}')


WeightWatcher spectral analysis using canonical schema-v2 scientific runs only.

In [ ]:
candidates = discover_pair_candidates(RESULTS_ROOT, include_legacy=INCLUDE_LEGACY)
canonical_pairs, pair_audit = select_canonical_pairs(candidates)
runs = discover_canonical_runs(RESULTS_ROOT, include_legacy=INCLUDE_LEGACY)
inventory = build_run_inventory(runs)
pair_audit.to_csv(ANALYSIS_DIR/'pair_candidate_audit.csv', index=False)
inventory.to_csv(ANALYSIS_DIR/'run_inventory.csv', index=False)
print(f'Canonical pairs: {len(canonical_pairs)}')
display(pair_audit)
display(inventory)


In [ ]:
specs = []
projs = []
metrics_frames = []

for r in runs:
    art = load_run_artifacts(Path(r['run_dir']))
    run_context = {
        'seed': r['seed'],
        'pair_id': r['pair_id'],
        'optimizer_family': r['optimizer_family'],
        'optimizer_label': r['optimizer_label'],
        'optimizer_raw': r['optimizer_raw'],
    }
    schema_version = r['manifest'].get('scientific_schema_version')
    specs.append(
        art['spectral'].assign(
            **run_context,
            valid_for_science=True,
            scientific_schema_version=schema_version,
        )
    )
    metrics_frames.append(art['metrics'].assign(**run_context))
    p = art['projection']
    if not p.empty:
        projs.append(p.assign(**run_context))

spectral = pd.concat(specs, ignore_index=True)
metrics = pd.concat(metrics_frames, ignore_index=True) if metrics_frames else pd.DataFrame()
projections = pd.concat(projs, ignore_index=True) if projs else pd.DataFrame()

spectral.to_csv(ANALYSIS_DIR / 'spectral_records_scientific.csv', index=False)
metrics.to_csv(ANALYSIS_DIR / 'metrics_records_scientific.csv', index=False)
projections.to_csv(ANALYSIS_DIR / 'projection_records_scientific.csv', index=False)

schema_numeric = pd.to_numeric(spectral['scientific_schema_version'], errors='coerce')
valid = spectral[
    spectral.spectral_estimator.astype(str).str.lower().eq('weightwatcher')
    & spectral.valid_for_science.astype(bool)
    & schema_numeric.ge(2)
].copy()

validity = (
    spectral.groupby(['optimizer_raw', 'seed'], dropna=False)
    .agg(
        rows=('alpha', 'size'),
        weightwatcher_rows=('valid_weightwatcher', 'sum'),
        finite_alpha_rows=('alpha', lambda s: pd.to_numeric(s, errors='coerce').notna().sum()),
        finite_D_rows=('D', lambda s: pd.to_numeric(s, errors='coerce').notna().sum() if 'D' in spectral else 0),
        finite_num_evals_rows=('num_evals', lambda s: pd.to_numeric(s, errors='coerce').notna().sum() if 'num_evals' in spectral else 0),
        layer_groups=('layer_group', lambda s: ', '.join(sorted(map(str, pd.Series(s).dropna().unique())))),
    )
    .reset_index()
)
validity['weightwatcher_row_fraction'] = validity['weightwatcher_rows'] / validity['rows'].replace(0, np.nan)
validity.to_csv(ANALYSIS_DIR / 'spectral_validity_audit.csv', index=False)
display(validity)

proj_layers = valid[valid.layer_group == 'projected_transformer_matrix'].copy()
proj_layers['alpha'] = pd.to_numeric(proj_layers['alpha'], errors='coerce')
proj_layers['alpha_distance_to_2'] = (proj_layers['alpha'] - 2).abs()
for col in ['D', 'num_evals', 'spectral_norm', 'stable_rank', 'log_alpha_norm', 'alpha_weighted']:
    if col in proj_layers:
        proj_layers[col] = pd.to_numeric(proj_layers[col], errors='coerce')

summary = (
    proj_layers.groupby(['optimizer_family', 'seed', 'tokens_seen'], dropna=False)
    .agg(
        average_alpha=('alpha', 'mean'),
        median_alpha=('alpha', 'median'),
        std_alpha=('alpha', 'std'),
        q25_alpha=('alpha', lambda s: s.quantile(.25)),
        q75_alpha=('alpha', lambda s: s.quantile(.75)),
        median_alpha_distance_to_2=('alpha_distance_to_2', 'median'),
        eligible_matrices=('alpha', 'count'),
    )
    .reset_index()
)
summary.to_csv(ANALYSIS_DIR / 'run_snapshot_alpha_summary.csv', index=False)

layer_summary = (
    proj_layers.groupby(['optimizer_family', 'layer_name', 'tokens_seen'], dropna=False)
    .agg(
        median_alpha=('alpha', 'median'),
        q25=('alpha', lambda s: s.quantile(.25)),
        q75=('alpha', lambda s: s.quantile(.75)),
        median_D=('D', 'median') if 'D' in proj_layers else ('alpha', 'size'),
        median_num_evals=('num_evals', 'median') if 'num_evals' in proj_layers else ('alpha', 'size'),
    )
    .reset_index()
)
layer_summary.to_csv(ANALYSIS_DIR / 'projected_layer_alpha_summary.csv', index=False)

saved_plots = []

def savefig(fig, filename, *, show=True):
    path = ANALYSIS_DIR / filename
    fig.tight_layout()
    fig.savefig(path, dpi=160, bbox_inches='tight')
    if show:
        display(fig)
    plt.close(fig)
    saved_plots.append(path.name)


if not proj_layers.empty:
    layer_step = (
        proj_layers.dropna(subset=['tokens_seen', 'alpha'])
        .groupby(['optimizer_family', 'tokens_seen', 'layer_name'], dropna=False)['alpha']
        .median()
        .reset_index()
        .sort_values(['optimizer_family', 'layer_name', 'tokens_seen'])
    )
    layer_step.to_csv(ANALYSIS_DIR / 'layer_alpha_per_step.csv', index=False)

    optimizer_families = list(layer_step['optimizer_family'].dropna().unique())
    ncols = min(2, max(1, len(optimizer_families)))
    nrows = int(np.ceil(len(optimizer_families) / ncols))
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(8 * ncols, 4.5 * nrows), squeeze=False, sharex=True, sharey=True)
    for ax in axes.flat[len(optimizer_families):]:
        ax.axis('off')
    for ax, fam in zip(axes.flat, optimizer_families):
        g = layer_step[layer_step['optimizer_family'].eq(fam)]
        for layer_name, gg in g.groupby('layer_name', sort=True):
            ax.plot(gg['tokens_seen'], gg['alpha'], marker='o', markersize=2.5, linewidth=1, alpha=.7, label=layer_name)
        ax.axhline(2, color='black', linestyle='--', linewidth=1, alpha=.5)
        ax.set_title(f"{OPTIMIZER_LABELS.get(fam, fam)} layer alphas per step")
        ax.set_xlabel('tokens_seen')
        ax.set_ylabel('WeightWatcher alpha')
        ax.grid(True, alpha=.2)
        if g['layer_name'].nunique() <= 16:
            ax.legend(fontsize=6, ncol=2)
    savefig(fig, 'layer_alpha_per_step.png')

for y, filename, ylabel in [
    ('alpha', 'alpha_trajectories_by_layer.png', 'WeightWatcher alpha'),
    ('median_alpha', 'median_projected_alpha_vs_tokens.png', 'median projected-layer alpha'),
    ('alpha_distance_to_2', 'distance_to_alpha_2_vs_tokens.png', '|alpha - 2|'),
]:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    data = proj_layers if y != 'median_alpha' else summary
    for fam, g in data.groupby('optimizer_family'):
        color = OPTIMIZER_COLORS.get(fam)
        label = OPTIMIZER_LABELS.get(fam, fam)
        group_cols = ['seed'] if y == 'median_alpha' else ['seed', 'layer_name']
        for _, gg in g.sort_values('tokens_seen').groupby(group_cols):
            ax.plot(gg.tokens_seen, gg[y], alpha=.18, color=color)
        mean = g.groupby('tokens_seen')[y].mean().sort_index()
        ax.plot(mean.index, mean.values, lw=3, label=label, color=color)
    ax.legend()
    ax.set_xlabel('tokens_seen')
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel + ' over training')
    savefig(fig, filename)


if not summary.empty and 'average_alpha' in summary:
    average_alpha_bands = (
        summary.dropna(subset=['tokens_seen', 'average_alpha'])
        .groupby(['optimizer_family', 'tokens_seen'], dropna=False)
        .agg(
            average_alpha=('average_alpha', 'mean'),
            alpha_std=('average_alpha', 'std'),
            seed_count=('average_alpha', 'count'),
        )
        .reset_index()
        .sort_values(['optimizer_family', 'tokens_seen'])
    )
    average_alpha_bands['alpha_std'] = average_alpha_bands['alpha_std'].fillna(0)
    average_alpha_bands['mean_minus_2std'] = average_alpha_bands['average_alpha'] - 2 * average_alpha_bands['alpha_std']
    average_alpha_bands['mean_plus_2std'] = average_alpha_bands['average_alpha'] + 2 * average_alpha_bands['alpha_std']
    average_alpha_bands.to_csv(ANALYSIS_DIR / 'average_alpha_mean_std_bands.csv', index=False)

    fig, ax = plt.subplots(figsize=(8, 4.5))
    for fam, g in average_alpha_bands.groupby('optimizer_family'):
        g = g.sort_values('tokens_seen')
        x = g['tokens_seen'].to_numpy(dtype=float)
        mean = g['average_alpha'].to_numpy(dtype=float)
        lower = g['mean_minus_2std'].to_numpy(dtype=float)
        upper = g['mean_plus_2std'].to_numpy(dtype=float)
        color = OPTIMIZER_COLORS.get(fam)
        label = OPTIMIZER_LABELS.get(fam, fam)
        ax.plot(x, mean, lw=3, label=f'{label} mean alpha', color=color)
        ax.fill_between(x, lower, upper, color=color, alpha=.18, label=f'{label} ±2σ band')
    ax.axhline(2, color='black', linestyle='--', linewidth=1, alpha=.5, label='alpha = 2')
    ax.set_xlabel('tokens_seen')
    ax.set_ylabel('average projected-layer alpha')
    ax.set_title('Average WeightWatcher alpha with mean ±2 seed-standard-deviation band')
    ax.grid(True, alpha=.2)
    ax.legend()
    savefig(fig, 'average_alpha_mean_std_band.png')

if not proj_layers.empty:
    alpha_heatmap = (
        proj_layers.pivot_table(
            index='layer_name',
            columns='tokens_seen',
            values='alpha',
            aggfunc='median',
        )
        .sort_index()
    )
    fig, ax = plt.subplots(figsize=(11, max(4, 0.28 * len(alpha_heatmap))))
    im = ax.imshow(alpha_heatmap, aspect='auto', interpolation='nearest', cmap='viridis')
    ax.set_yticks(range(len(alpha_heatmap.index)))
    ax.set_yticklabels(alpha_heatmap.index, fontsize=7)
    ax.set_xticks(range(len(alpha_heatmap.columns)))
    ax.set_xticklabels([f'{int(x):,}' for x in alpha_heatmap.columns], rotation=45, ha='right', fontsize=7)
    ax.set_xlabel('tokens_seen')
    ax.set_title('Median WeightWatcher alpha by projected layer')
    fig.colorbar(im, ax=ax, label='median alpha')
    savefig(fig, 'layer_alpha_heatmap.png')

    fig, ax = plt.subplots(figsize=(7, 4.5))
    for fam, g in proj_layers.groupby('optimizer_family'):
        ax.hist(g['alpha'].dropna(), bins=30, alpha=.45, label=OPTIMIZER_LABELS.get(fam, fam), color=OPTIMIZER_COLORS.get(fam))
    ax.axvline(2, color='black', linestyle='--', linewidth=1, label='target alpha = 2')
    ax.set_xlabel('WeightWatcher alpha')
    ax.set_ylabel('projected-layer records')
    ax.set_title('Distribution of projected-layer alphas')
    ax.legend()
    savefig(fig, 'layer_alpha_distribution.png')

metric_plot_specs = [
    ('D', 'spectral_fit_quality.png', 'D / KS statistic'),
    ('num_evals', 'num_evals_vs_tokens.png', 'num_evals'),
    ('spectral_norm', 'spectral_norm_vs_tokens.png', 'spectral norm'),
    ('stable_rank', 'stable_rank_vs_tokens.png', 'stable rank'),
    ('log_alpha_norm', 'log_alpha_norm_vs_tokens.png', 'log alpha norm'),
    ('alpha_weighted', 'alpha_weighted_vs_tokens.png', 'alpha weighted'),
]
for metric, filename, ylabel in metric_plot_specs:
    if metric not in proj_layers or proj_layers[metric].dropna().empty:
        continue
    fig, ax = plt.subplots(figsize=(8, 4.5))
    for fam, g in proj_layers.groupby('optimizer_family'):
        color = OPTIMIZER_COLORS.get(fam)
        label = OPTIMIZER_LABELS.get(fam, fam)
        for _, gg in g.sort_values('tokens_seen').groupby(['seed', 'layer_name']):
            ax.plot(gg.tokens_seen, gg[metric], alpha=.15, color=color)
        med = g.groupby('tokens_seen')[metric].median().sort_index()
        ax.plot(med.index, med.values, lw=3, label=label, color=color)
    ax.set_xlabel('tokens_seen')
    ax.set_ylabel(ylabel)
    ax.set_title(f'{ylabel} over training')
    ax.legend()
    savefig(fig, filename)

if not metrics.empty:
    for metric, ylabel in [('validation_loss', 'validation loss'), ('train_loss', 'train loss'), ('tokens_per_second', 'tokens/sec')]:
        if metric not in metrics or metrics[metric].dropna().empty:
            continue
        fig, ax = plt.subplots(figsize=(8, 4.5))
        for fam, g in metrics.groupby('optimizer_family'):
            color = OPTIMIZER_COLORS.get(fam)
            label = OPTIMIZER_LABELS.get(fam, fam)
            for _, gg in g.sort_values('tokens_seen').groupby('seed'):
                ax.plot(gg.tokens_seen, gg[metric], alpha=.25, color=color)
            med = g.groupby('tokens_seen')[metric].median().sort_index()
            ax.plot(med.index, med.values, lw=3, label=label, color=color)
        ax.set_xlabel('tokens_seen')
        ax.set_ylabel(ylabel)
        ax.set_title(f'{ylabel} run metric')
        ax.legend()
        savefig(fig, f'{metric}_run_metric.png')

fig, ax = plt.subplots(figsize=(8, 4.5))
for fam, g in summary.groupby('optimizer_family'):
    ax.plot(g.tokens_seen, g.median_alpha, '.', label=OPTIMIZER_LABELS.get(fam, fam), color=OPTIMIZER_COLORS.get(fam))
if not projections.empty and 'tokens_seen' in projections:
    for x in pd.to_numeric(projections.tokens_seen, errors='coerce').dropna().unique():
        ax.axvline(x, color='gray', alpha=.1)
ax.legend()
ax.set_xlabel('tokens_seen')
ax.set_ylabel('median projected-layer alpha')
ax.set_title('Projection event markers on alpha snapshots')
savefig(fig, 'projection_markers_on_alpha.png')

plot_manifest = pd.DataFrame({'plot_file': saved_plots})
plot_manifest.to_csv(ANALYSIS_DIR / 'weightwatcher_plot_manifest.csv', index=False)
display(plot_manifest)

print('Finite-size warning: level-0 width-64 style models can produce noisy layer-level power-law estimates; aggregate within run/snapshot before comparing seeds.')
